# Table Relationships

## Objective

Understand how the datasets are related to each other before performing
data preparation and exploratory data analysis.

The main goals are:
- identify the key columns used to connect tables;
- understand the relationship between tables;
- check the cardinality of important relationships; and
- check whether key values are consistent across related tables.

# 1. Load Dataset

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


data_dir = Path('../dataset')

In [17]:
sellers = pd.read_csv(data_dir / "olist_sellers_dataset.csv")
customers = pd.read_csv(data_dir / "olist_customers_dataset.csv")
products = pd.read_csv(data_dir / "olist_products_dataset.csv")
geolocation = pd.read_csv(data_dir / "olist_geolocation_dataset.csv")
orders = pd.read_csv(data_dir / "olist_orders_dataset.csv")
items = pd.read_csv(data_dir / "olist_order_items_dataset.csv")
payments = pd.read_csv(data_dir / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_dir / "olist_order_reviews_dataset.csv")
translated_products = pd.read_csv(data_dir / "product_category_name_translation.csv")


# 2. Table Relationship
| Table A     | Key                      | Table B        | Key                         | Relationship | Note |
| ----------- | ------------------------ | -------------- | --------------------------- | ------------ | ---- |
| orders      | order_id                 | payments       | order_id                    | 1:0..N       | 1 order tanpa payment |
| orders      | order_id                 | reviews        | order_id                    | 1:0..N       | 768 order tanpa review; `review_id` tidak unik |
| orders      | order_id                 | items          | order_id                    | 1:0..N       | 775 order tanpa item |
| orders      | customer_id              | customers      | customer_id                 | 1:1          | pelanggan sebenarnya = `customer_unique_id` |
| items       | product_id               | products       | product_id                  | N:1          | |
| items       | seller_id                | sellers        | seller_id                   | N:1          | |
| products    | product_category_name    | translated_products | product_category_name  | N:1          | lihat Relation 9 |
| sellers     | seller_zip_code_prefix   | geolocation    | geolocation_zip_code_prefix | N:N (mentah) | jadi N:1 setelah geolocation diagregasi per zip |
| customers   | customer_zip_code_prefix | geolocation    | geolocation_zip_code_prefix | N:N (mentah) | jadi N:1 setelah geolocation diagregasi per zip |


# 3. Key Identification

## Relation Number 1 (`Orders` <-> `Payments`)

In [18]:
# Find columnd in each table
print(orders.columns)
print(payments.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')


In [19]:
# Find the number of rows in each table
print("Orders number of rows: " + str(orders.shape[0]))
print("Payments number of rows: " + str(payments.shape[0]))

Orders number of rows: 99441
Payments number of rows: 103886


In [20]:
# Find the unique ID from the table Orders and Payments
orders_orderid_uniq = orders["order_id"].nunique()
payments_orderid_uniq = payments["order_id"].nunique()

print(f"Number of unique order_id in table Orders: {orders_orderid_uniq}")
print(f"Number of unique order_id in table Payments: {payments_orderid_uniq}")

Number of unique order_id in table Orders: 99441
Number of unique order_id in table Payments: 99440


In [21]:
# Find the reason why the result -1
orders["order_id"].isin(payments["order_id"]).value_counts()

order_id
True     99440
False        1
Name: count, dtype: int64

In [22]:
missing_payment_order = orders.loc[
    ~orders["order_id"].isin(payments["order_id"]),
    "order_id"
]

print(missing_payment_order)

30710    bfbd0f9bdef84302105ad712db648a6c
Name: order_id, dtype: str


The reason why the difference is 1 is because there's one `order_id` from ```Orders``` not recorded in ```Payments```

In [23]:
# Find the missing values from table Orders and Payments
orderid_missvls_orders = orders["order_id"].isna().sum()
orderid_missvls_payments = payments["order_id"].isna().sum()

print(f"Number of missing values in table Orders: {orderid_missvls_orders}")
print(f"Number of missing values in table Payments: {orderid_missvls_payments}")

Number of missing values in table Orders: 0
Number of missing values in table Payments: 0


In [24]:
# Find the number of duplicates order_id in each table
print("Duplicated values in Orders dataset: " + str(orders["order_id"].duplicated().sum()))
print("Duplicated values in Payments dataset: " + str(payments["order_id"].duplicated().sum()))

Duplicated values in Orders dataset: 0
Duplicated values in Payments dataset: 4446


In [25]:
# Number of orders with more than 1 row, and orders with no row, in Payments
rows_per_order = payments.groupby("order_id").size()
print(f"Orders with more than 1 payment row: {(rows_per_order > 1).sum()}")
print(f"Orders with 0 payment row: {(~orders['order_id'].isin(payments['order_id'])).sum()}")

Orders with more than 1 payment row: 2961
Orders with 0 payment row: 1


Hanya sebagian `order_id` yang muncul lebih dari satu kali pada ```Payments``` (4.446 baris kelebihan; jumlah order-nya ada di sel di atas), dan 1 `order_id` pada ```Orders``` tidak memiliki payment sama sekali.

Kesimpulan: 1:0..N (satu order punya nol sampai banyak payment). `INNER JOIN` membuang order tanpa payment, dan join langsung ke tabel lain menggandakan baris.

## Relation Number 2 (`Orders` <-> `Reviews`)

In [26]:
# Find column in each table
print(orders.columns)
print(reviews.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')


In [27]:
# Find the number of rows in each table
print("Orders number of rows: " + str(orders.shape[0]))
print("Reviews number of rows: " + str(reviews.shape[0]))

Orders number of rows: 99441
Reviews number of rows: 99224


In [28]:
# Find the unique ID from the table Orders and Reviews
reviews_orderid_uniq = reviews["order_id"].nunique()

print(f"Number of unique order_id in table Orders: {orders_orderid_uniq}")
print(f"Number of unique order_id in table Reviews: {reviews_orderid_uniq}")

Number of unique order_id in table Orders: 99441
Number of unique order_id in table Reviews: 98673


In [29]:
orders["order_id"].isin(reviews["order_id"]).value_counts()

order_id
True     98673
False      768
Name: count, dtype: int64

In [30]:
missing_reviews_order = orders.loc[
    ~orders["order_id"].isin(reviews["order_id"]),
    "order_id"
]

print(missing_reviews_order)

16       403b97836b0c04a622354cf531062e5f
154      6942b8da583c2f9957e990d028607019
311      4906eeadde5f70b308c20c4a8f20be02
382      b7a4a9ecb1cd3ef6a3e36a48e200e3be
390      59b32faedc12322c672e95ec3716d614
                       ...               
98697    0c384d67524b5b92aa2fa6c8baa9a983
98781    906a6b0a96d89ee226e4977e99b80b9e
99007    5333db16fe357175d39c82840dd3269d
99134    2f2df159f26ddb73d55ee72372200d3e
99245    dff2b9b8d7cfc595836945e1443789c3
Name: order_id, Length: 768, dtype: str


ini mengindikasikan bahwa tidak semua `order_id` di `Orders` meninggalkan reviews pada `Reviews`

In [31]:
# Find the missing values from table Orders and Reviews
orderid_missvls_reviews = reviews["order_id"].isna().sum()

print(f"Number of missing values in table Orders: {orderid_missvls_orders}")
print(f"Number of missing values in table Reviews: {orderid_missvls_reviews}")

Number of missing values in table Orders: 0
Number of missing values in table Reviews: 0


In [32]:
# Find the number of duplicates order_id in each table
print("Duplicated values in Orders dataset: " + str(orders["order_id"].duplicated().sum()))
print("Duplicated values in Reviews dataset: " + str(reviews["order_id"].duplicated().sum()))

Duplicated values in Orders dataset: 0
Duplicated values in Reviews dataset: 551


In [33]:
# Number of orders with more than 1 row, and orders with no row, in Reviews
rows_per_order = reviews.groupby("order_id").size()
print(f"Orders with more than 1 review row: {(rows_per_order > 1).sum()}")
print(f"Orders with 0 review row: {(~orders['order_id'].isin(reviews['order_id'])).sum()}")

Orders with more than 1 review row: 547
Orders with 0 review row: 768


Hanya sebagian `order_id` yang muncul lebih dari satu kali pada ```Reviews``` (551 baris kelebihan), dan 768 `order_id` pada ```Orders``` tidak memiliki review. Selain itu `review_id` juga tidak unik (814 duplikat), jadi grain ```Reviews``` bukan per `review_id`.

Kesimpulan: 1:0..N (satu order punya nol sampai banyak review).

## Relation Number 3 (`Orders` <-> `Items`)

In [34]:
# Find columnd in each table
print(orders.columns)
print(items.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')


In [35]:
# Find the number of rows in each table
print("Orders number of rows: " + str(orders.shape[0]))
print("Items number of rows: " + str(items.shape[0]))

Orders number of rows: 99441
Items number of rows: 112650


In [36]:
# Find the unique ID from the table Orders and Items
items_orderid_uniq = items["order_id"].nunique()

print(f"Number of unique order_id in table Orders: {orders_orderid_uniq}")
print(f"Number of unique order_id in table Items: {items_orderid_uniq}")

Number of unique order_id in table Orders: 99441
Number of unique order_id in table Items: 98666


In [37]:
# # Plot
# x = np.arange(2)
# width = 0.35

# plt.figure(figsize=(8, 5))

# plt.bar(x - width/2, [orders.shape[0], items.shape[0]], width, label="Number of Rows")
# plt.bar(x + width/2, [orders_orderid_uniq, items_orderid_uniq], width, label="Unique order_id")

# plt.xticks(x, ["Orders", "Items"])
# plt.ylabel("Count")
# plt.title("Number of Rows vs Unique order_id")
# plt.legend()

# plt.tight_layout()
# plt.show()

In [38]:
orders['order_id'].isin(items['order_id']).value_counts()

order_id
True     98666
False      775
Name: count, dtype: int64

In [39]:
missing_items_order = orders.loc[
    ~orders["order_id"].isin(items["order_id"]),
    "order_id"
]

print(missing_items_order)

266      8e24261a7e58791d10cb1bf9da94df5c
586      c272bcd21c287498b4883c7512019702
687      37553832a3a89c9b2db59701c357ca67
737      d57e15fb07fd180f06ab3926b39edcd2
1130     00b1cb0320190ca0daa2c88b35206009
                       ...               
99252    aaab15da689073f8f9aa978a390a69d1
99283    3a3cddda5a7c27851bd96c3313412840
99347    a89abace0dcc01eeb267a9660b5ac126
99348    a69ba794cc7deb415c3e15a0a3877e69
99415    5fabc81b6322c8443648e1b21a6fef21
Name: order_id, Length: 775, dtype: str


ini mengindikasikan bahwa tidak semua `order_id` yang tercatat di `Orders` memiliki record pada `Items`.

In [40]:
# Find the missing values from table Orders and Items
orderid_missvls_items = items["order_id"].isna().sum()

print(f"Number of missing values in table Orders: {orderid_missvls_orders}")
print(f"Number of missing values in table Items: {orderid_missvls_items}")

Number of missing values in table Orders: 0
Number of missing values in table Items: 0


In [41]:
# Find the number of duplicates order_id in each table
print("Duplicated values in Orders dataset: " + str(orders["order_id"].duplicated().sum()))
print("Duplicated values in Items dataset: " + str(items["order_id"].duplicated().sum()))

Duplicated values in Orders dataset: 0
Duplicated values in Items dataset: 13984


In [42]:
# Number of orders with more than 1 row, and orders with no row, in Items
rows_per_order = items.groupby("order_id").size()
print(f"Orders with more than 1 item row: {(rows_per_order > 1).sum()}")
print(f"Orders with 0 item row: {(~orders['order_id'].isin(items['order_id'])).sum()}")

Orders with more than 1 item row: 9803
Orders with 0 item row: 775


Hanya sebagian `order_id` yang muncul lebih dari satu kali pada ```Items``` (13.984 baris kelebihan), dan 775 `order_id` pada ```Orders``` tidak memiliki item.

Kesimpulan: 1:0..N (satu order punya nol sampai banyak item).

## Relation Number 4 (`Orders` <-> `Customers`)

In [43]:
# Find columnd in each table
print(orders.columns)
print(customers.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')


In [44]:
# Find the number of rows in each table
print("Orders number of rows: " + str(orders.shape[0]))
print("Customers number of rows: " + str(customers.shape[0]))

Orders number of rows: 99441
Customers number of rows: 99441


In [45]:
# Find the unique ID from the table Orders and Customers
orders_custid_uniq = orders["customer_id"].nunique()
customers_custid_uniq = customers["customer_id"].nunique()

print(f"Number of unique customer_id in table Orders: {orders_custid_uniq}")
print(f"Number of unique customer_id in table Customers: {customers_custid_uniq}")

Number of unique customer_id in table Orders: 99441
Number of unique customer_id in table Customers: 99441


In [46]:
orders['customer_id'].isin(customers['customer_id']).value_counts()

customer_id
True    99441
Name: count, dtype: int64

In [47]:
# Find the missing values from table Orders and Customers
custid_missvls_orders = orders["customer_id"].isna().sum()
custid_missvls_customer = customers["customer_id"].isna().sum()

print(f"Number of missing values in table Orders: {custid_missvls_orders}")
print(f"Number of missing values in table Customers: {custid_missvls_customer}")

Number of missing values in table Orders: 0
Number of missing values in table Customers: 0


In [48]:
# Find the number of duplicates customer_id in each table
print("Duplicated values in Orders dataset: " + str(orders["customer_id"].duplicated().sum()))
print("Duplicated values in Customers dataset: " + str(customers["customer_id"].duplicated().sum()))

Duplicated values in Orders dataset: 0
Duplicated values in Customers dataset: 0


In [49]:
# customer_id is unique per order; customer_unique_id identifies the real customer
orders_per_customer = (orders.merge(customers, on="customer_id")
                       .groupby("customer_unique_id").size())

print(f"Unique customers (customer_unique_id): {orders_per_customer.shape[0]}")
print(f"Customers with more than 1 order: {(orders_per_customer > 1).sum()}")

Unique customers (customer_unique_id): 96096
Customers with more than 1 order: 2997


`customer_id` unik di ```Orders``` maupun ```Customers``` dan semuanya cocok, jadi relasinya 1:1 (bukan N:1). Pelanggan yang sebenarnya diidentifikasi oleh `customer_unique_id`, yang bisa muncul pada lebih dari satu `customer_id`.

Kesimpulan: 1:1 pada `customer_id`; untuk menghitung jumlah pelanggan gunakan `customer_unique_id`.

## Relation Number 5 (`items` <-> `products`)

In [50]:
# Find columnd in each table
print(items.columns)
print(products.columns)

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')


In [51]:
# Find the number of rows in each table
print("Items number of rows: " + str(items.shape[0]))
print("Products number of rows: " + str(products.shape[0]))

Items number of rows: 112650
Products number of rows: 32951


In [52]:
# Find the unique ID from the table Items and Products
items_prodid_uniq = items["product_id"].nunique()
products_prodid_uniq = products["product_id"].nunique()

print(f"Number of unique product_id in table Items: {items_prodid_uniq}")
print(f"Number of unique product_id in table Products: {products_prodid_uniq}")

Number of unique product_id in table Items: 32951
Number of unique product_id in table Products: 32951


In [53]:
products['product_id'].isin(items['product_id']).value_counts()

product_id
True    32951
Name: count, dtype: int64

In [54]:
# Check the other direction: every product_id in Items must exist in Products
items['product_id'].isin(products['product_id']).value_counts()

product_id
True    112650
Name: count, dtype: int64

In [55]:
# Find the missing values from table Items and Products
prodid_missvls_items = items["product_id"].isna().sum()
prodid_missvls_products = products["product_id"].isna().sum()

print(f"Number of missing values in table Items: {prodid_missvls_items}")
print(f"Number of missing values in table Products: {prodid_missvls_products}")

Number of missing values in table Items: 0
Number of missing values in table Products: 0


In [56]:
# Find the number of duplicates product_id in each table
print("Duplicated values in Items dataset: " + str(items["product_id"].duplicated().sum()))
print("Duplicated values in Products dataset: " + str(products["product_id"].duplicated().sum()))

Duplicated values in Items dataset: 79699
Duplicated values in Products dataset: 0


Setiap `product_id` pada ```Products``` ada di ```Items``` (32.951 unik di kedua tabel), dan satu produk bisa muncul di banyak item.

Kesimpulan: N:1 (many to one)

## Relation Number 6 (`Items` <-> `Sellers`)

In [57]:
# Find columnd in each table
print(items.columns)
print(sellers.columns)

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')
Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='str')


In [58]:
# Find the number of rows in each table
print("Items number of rows: " + str(items.shape[0]))
print("Sellers number of rows: " + str(sellers.shape[0]))

Items number of rows: 112650
Sellers number of rows: 3095


In [59]:
# Find the unique ID from the table Items and Sellers
items_sellerid_uniq = items["seller_id"].nunique()
sellers_sellerid_uniq = sellers["seller_id"].nunique()

print(f"Number of unique seller_id in table Items: {items_sellerid_uniq}")
print(f"Number of unique seller_id in table Sellers: {sellers_sellerid_uniq}")

Number of unique seller_id in table Items: 3095
Number of unique seller_id in table Sellers: 3095


In [60]:
items['seller_id'].isin(sellers['seller_id']).value_counts()

seller_id
True    112650
Name: count, dtype: int64

In [61]:
# Find the missing values from table Items and Sellers
sellerid_missvls_items = items["seller_id"].isna().sum()
sellerid_missvls_sellers = sellers["seller_id"].isna().sum()

print(f"Number of missing values in table Items: {sellerid_missvls_items}")
print(f"Number of missing values in table Sellers: {sellerid_missvls_sellers}")

Number of missing values in table Items: 0
Number of missing values in table Sellers: 0


In [62]:
# Find the number of duplicates seller_id in each table
print("Duplicated values in Items dataset: " + str(items["seller_id"].duplicated().sum()))
print("Duplicated values in Sellers dataset: " + str(sellers["seller_id"].duplicated().sum()))

Duplicated values in Items dataset: 109555
Duplicated values in Sellers dataset: 0


Semua 112.650 baris ```Items``` memiliki `seller_id` yang ada di ```Sellers``` (3.095 unik di kedua tabel).

Kesimpulan: N:1 (many to one)

## Relation Number 7 (`Sellers` <-> `Geolocation`)

In [63]:
# Find column in each table
print(sellers.columns)
print(geolocation.columns)

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='str')
Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str')


In [64]:
# Find the number of rows in each table
print("Sellers number of rows: " + str(sellers.shape[0]))
print("Geolocation number of rows: " + str(geolocation.shape[0]))

Sellers number of rows: 3095
Geolocation number of rows: 1000163


In [65]:
# Find the number of unique zip code prefix in each table
sellers_zipcode_uniq = sellers["seller_zip_code_prefix"].nunique()
geolocation_zipcode_uniq = geolocation["geolocation_zip_code_prefix"].nunique()

print(f"Number of unique zip code prefix in table Sellers: {sellers_zipcode_uniq}")
print(f"Number of unique zip code prefix in table Geolocation: {geolocation_zipcode_uniq}")

Number of unique zip code prefix in table Sellers: 2246
Number of unique zip code prefix in table Geolocation: 19015


In [66]:
sellers['seller_zip_code_prefix'].sort_values(ascending=False)

187     99730
1850    99700
1704    99670
2105    99500
1458    99500
        ...  
2332     1023
1567     1022
2021     1021
599      1021
848      1001
Name: seller_zip_code_prefix, Length: 3095, dtype: int64

In [67]:
geolocation['geolocation_zip_code_prefix'].sort_values(ascending=False)

999758     99990
999864     99990
1000089    99980
1000048    99980
1000026    99980
           ...  
1062        1001
596         1001
985         1001
639         1001
608         1001
Name: geolocation_zip_code_prefix, Length: 1000163, dtype: int64

In [68]:
# Seller zip code prefixes that do not exist in Geolocation
missing_seller_zip = ~sellers["seller_zip_code_prefix"].isin(geolocation["geolocation_zip_code_prefix"])

print(f"Sellers whose zip code is not in Geolocation: {missing_seller_zip.sum()}")
print(f"Unique zip codes not in Geolocation: {sellers.loc[missing_seller_zip, 'seller_zip_code_prefix'].nunique()}")

Sellers whose zip code is not in Geolocation: 7
Unique zip codes not in Geolocation: 7


In [69]:
# Number of rows per zip code prefix in Geolocation
geolocation.groupby("geolocation_zip_code_prefix").size().describe()

count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
dtype: float64

`Geolocation` memiliki 1.000.163 baris untuk 19.015 zip code prefix, sehingga satu zip muncul di banyak baris. Join langsung dari ```Sellers``` ke ```Geolocation``` akan menggandakan baris seller, jadi relasi "N:N" ini hanya berlaku pada data mentah.

Kesimpulan: `Geolocation` harus diagregasi menjadi 1 baris per zip code prefix terlebih dulu. Setelah itu relasinya N:1.

## Relation Number 8 (`Customers` <-> `Geolocation`)

In [70]:
# Customer zip code prefixes that do not exist in Geolocation
missing_customer_zip = ~customers["customer_zip_code_prefix"].isin(geolocation["geolocation_zip_code_prefix"])

print(f"Customers whose zip code is not in Geolocation: {missing_customer_zip.sum()}")
print(f"Unique zip codes not in Geolocation: {customers.loc[missing_customer_zip, 'customer_zip_code_prefix'].nunique()}")

Customers whose zip code is not in Geolocation: 278
Unique zip codes not in Geolocation: 157


Sama seperti Relation 7: `Geolocation` harus diagregasi per zip code prefix sebelum di-join. Zip yang tidak ditemukan di ```Geolocation``` (jumlahnya ada di sel di atas) akan menghasilkan koordinat kosong setelah `LEFT JOIN`.

## Relation Number 9 (`Products` <-> `Translated Products`)

In [71]:
# Product categories without English translation
cat_products = set(products["product_category_name"].dropna().unique())
cat_translation = set(translated_products["product_category_name"].unique())

print(f"Unique categories in Products: {len(cat_products)}")
print(f"Unique categories in Translated Products: {len(cat_translation)}")
print(f"Categories in Products without translation: {sorted(cat_products - cat_translation)}")
print(f"Categories in Translated Products not used by Products: {sorted(cat_translation - cat_products)}")
print(f"Products with missing category: {products['product_category_name'].isna().sum()}")

Unique categories in Products: 73
Unique categories in Translated Products: 71
Categories in Products without translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Categories in Translated Products not used by Products: []
Products with missing category: 610


`product_category_name` pada ```Products``` memiliki 73 kategori unik, sedangkan ```Translated Products``` hanya 71, dan 610 produk tidak memiliki kategori. Kategori tanpa terjemahan serta kategori kosong akan menjadi `NULL` jika di-join dengan `LEFT JOIN`, sehingga perlu aturan penanganan (misalnya fallback ke nama Portugis, dan label `unknown` untuk kategori kosong).

Kesimpulan: N:1 (many to one), dengan 2 kategori tanpa terjemahan.